# Tổng quan Milestone 1: Thu thập dữ liệu đa phiên bản trên 25.000 bài báo

## 1. Mục tiêu và quy trình thu thập 4 giai đoạn

Milestone 1 xây dựng quy trình thu thập dữ liệu cho toàn bộ 25.000 bài báo của 5 thành viên (mỗi thành viên phụ trách một dải 5.000 mã arXiv liên tiếp trong giai đoạn từ tháng 06/2024 đến tháng 02/2025). Quy trình tách biệt hai cấp độ thống kê theo đúng yêu cầu của đề bài:
- **Cấp bài báo (`publication`)**: lưu trữ thông tin định danh tổng hợp (`title`, `authors`, `submission_date`, `revised_dates`, `venue`) và danh sách tài liệu tham khảo thu thập từ Semantic Scholar.
- **Cấp phiên bản (`(paper, version)`)**: tải gói nguồn `/src/<id>vN`, lọc bỏ các tệp hình ảnh/nhị phân ngay trên luồng giải nén và kiểm toán độc lập toàn bộ lịch sử phiên bản từ v1 đến vN.

```{note}
**Hoàn tất 100% dải bài báo trên Google Colab CPU-only trước khi phân tích sang Milestone 2:**
Ở lượt chạy đầu, phân vùng của thành viên TRI (24120470) dừng ở mức 4.425/5.000 bài báo do chạm giới hạn thời gian 11,5 giờ của phiên máy ảo. Nhóm đã thực thi phiên chạy tiếp nối (`--resume`) trên Google Colab CPU-only để thu thập đủ 575 bài còn lại cùng 69 bài chờ xử lý, hoàn tất 5.000/5.000 bài và xác minh 8.338/8.338 phiên bản nguồn trước khi hợp nhất toàn nhóm.
```

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
draw_pipeline_diagram(
    "Hình M1.0. Sơ đồ quy trình thu thập dữ liệu 4 giai đoạn của Milestone 1",
    [
        ("1. Khám phá danh mục", "Atom XML lô 50-100 mã\nkết hợp OAI-PMH mới\n25.000 bài, 41.498 bản"),
        ("2. Tải và lọc nguồn", "export.arxiv.org/src/<id>vN\nGiãn cách tối thiểu 3,0 giây\nLọc ảnh giảm 89,30%"),
        ("3. Trích dẫn S2", "POST /paper/batch (50 bài)\nvà phân trang limit=1000\n976.351 dòng, 0 mồ côi"),
        ("4. Kiểm toán và xuất bảng", "1.713 gói sao lưu SHA-256\nHợp nhất 5 phân vùng\npapers và references"),
    ],
)

In [ ]:
df_m1_1 = pd.DataFrame([
    ("Tổng số bài báo được phân công (5 thành viên)", scope["group_assigned"], "bài báo"),
    ("Số bài báo hợp nhất duy nhất (không trùng chéo)", scope["papers"], "bài báo"),
    ("Tổng số phiên bản ghi nhận (v1 đến vN)", scope["versions"], "phiên bản"),
    ("Số gói sao lưu gia tăng có kiểm tra mã băm SHA-256", scope["archives_used"], "gói archive"),
    ("Tổng số dòng trích dẫn tầng vận hành (working)", m1["working_reference_rows"], "cạnh trích dẫn"),
    ("Số cạnh trích dẫn có đích arXiv (bảng references chuẩn)", m1["arxiv_reference_rows"], "cạnh trích dẫn"),
], columns=["Chỉ tiêu tổng hợp", "Số lượng", "Đơn vị"])
display(style_table(df_m1_1, caption="Bảng M1.1. Quy mô kho dữ liệu hợp nhất của 5 thành viên sau khử trùng.", int_cols=["Số lượng"]))

In [ ]:
sa = m1["source_audit"]
metrics = pd.DataFrame([
    ("Phiên bản có mã nguồn TeX xác minh (tex_verified)", sa.get("tex_verified", 0), scope["versions"]),
    ("Phiên bản đã kiểm toán hợp lệ (gồm cả bản chỉ có PDF)", m1["source_verified"], scope["versions"]),
    ("Bài báo xác minh hợp lệ đủ 100% phiên bản", m1["papers_with_all_sources_verified"], scope["papers"]),
    ("Bài báo có phản hồi S2 xác minh (success hoặc empty)", m1["verified_reference_papers"], scope["papers"]),
    ("Bài báo hoàn tất cả kiểm toán nguồn và trích dẫn S2", m1["papers_with_sources_and_s2_verified"], scope["papers"]),
], columns=["Chỉ tiêu độ phủ", "Số lượng đạt", "Mẫu số"])
metrics["Tỷ lệ (%)"] = 100.0 * metrics["Số lượng đạt"] / metrics["Mẫu số"]
display(style_table(
    metrics,
    caption="Bảng M1.2. Độ phủ kiểm toán nguồn và tài liệu tham khảo trên 25.000 bài báo.",
    int_cols=["Số lượng đạt", "Mẫu số"],
    pct_cols=["Tỷ lệ (%)"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8), gridspec_kw={"width_ratios": [1.32, 1.0]})
y_pos = np.arange(len(metrics))
bars = ax1.barh(y_pos, metrics["Tỷ lệ (%)"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], height=0.55)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([str(v) for v in metrics["Chỉ tiêu độ phủ"]], fontsize=8.4)
ax1.invert_yaxis()
ax1.set_xlim(0, 118)
ax1.grid(axis="x")
ax1.set_xlabel("Tỷ lệ đạt (%)")
ax1.set_title("Hình M1.1a. Tỷ lệ đạt trên 5 chỉ tiêu độ phủ M1")
for b, r in zip(bars, metrics.to_dict("records")):
    ax1.text(r["Tỷ lệ (%)"] + 1.2, b.get_y() + b.get_height() / 2,
             f"{fmt_int(r['Số lượng đạt'])} ({fmt_pct(r['Tỷ lệ (%)'])})", va="center", fontsize=8.0)

ssb = pd.Series(m1["source_status_breakdown"]).sort_values(ascending=True)
ssb_lbl = {
    "success": "Đủ TeX mọi phiên bản",
    "pdf_only": "Chỉ nộp PDF (pdf_only)",
    "partial_success": "Có TeX một phần",
    "not_found": "HTTP 404 trên arXiv",
    "no_latex": "Gói nguồn không có TeX",
    "error": "Chờ hoặc lỗi tải nguồn",
}
y_s = np.arange(len(ssb))
ax2.barh(y_s, ssb.to_numpy(dtype=float), color=BOOK_PALETTE["steel"], height=0.55)
ax2.set_yticks(y_s)
ax2.set_yticklabels([ssb_lbl.get(str(k), str(k)) for k in ssb.index], fontsize=8.4)
ax2.set_xscale("symlog", linthresh=50)
ax2.set_xlim(0, 80000)
ax2.grid(axis="x")
ax2.set_xlabel("Số bài báo (thang symlog)")
ax2.set_title("Hình M1.1b. Phân loại nguồn theo bài báo")
for i, v in enumerate(ssb.values):
    ax2.text(float(v) * 1.15 + 2, i, fmt_int(v), va="center", fontsize=8.0)
plt.tight_layout()
plt.show()

In [ ]:
pm1 = pd.DataFrame(m1["per_member"])
view_m1 = pd.DataFrame({
    "Thành viên": pm1["member"],
    "Dải mã arXiv": pm1["id_range"],
    "Số bài": pm1["publications"],
    "Phiên bản": pm1["versions"],
    "Có TeX": pm1["tex_verified"],
    "Chỉ PDF": pm1["empty_verified"],
    "Trích dẫn S2": pm1["working_references"],
    "Đích arXiv": pm1["course_references"],
    "Giảm ảnh (%)": 100.0 * pm1["source_reduction_rate"],
})
display(style_table(
    view_m1,
    caption="Bảng M1.3. Kết quả thu thập Milestone 1 theo từng thành viên (mỗi phân vùng 5.000 bài, không trùng chéo).",
    int_cols=["Số bài", "Phiên bản", "Có TeX", "Chỉ PDF", "Trích dẫn S2", "Đích arXiv"],
    pct_cols=["Giảm ảnh (%)"],
))

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/main.py).
- Mã nguồn triển khai: [`src/crawler/merge_shards.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/merge_shards.py).
- Mã nguồn triển khai: [`src/crawler/validate_sources.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/validate_sources.py).